# Production calibrators
Train and evaluate production-ready calibrators (affine + vector scaling) on UXM deconvolution predictions.

In [1]:
import json
from functools import reduce
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from syto.deconvolution.evaluation import compute_deconvolution_metrics
from syto.calibration.linear_calibrator import LinearCalibrator
from syto.calibration.vector_scaling_calibrator import VectorScalingCalibrator
from syto.cross_validation_engine import CrossValidationEngine

%load_ext autoreload
%autoreload 2

## Data loading

In [2]:
with open("../App/labels_dict.json", "r") as f:
    labels_to_ctype_names = json.load(f)
labels_to_ctype_names = {int(k): v for k, v in labels_to_ctype_names.items()}
ctype_names_to_labels = {v: k for k, v in labels_to_ctype_names.items()}
ctype_names_list = list(ctype_names_to_labels.keys())
print(f"{len(ctype_names_list)} cell types: {ctype_names_list[:5]} ...")

39 cell types: ['Adipocytes', 'Bladder-Ep', 'Blood-B', 'Blood-Granul', 'Blood-Mono+Macro'] ...


In [3]:
uxm_pred_folder = Path("../Data/training_data/mixture_predictions/uxm")

target_prop = np.load(uxm_pred_folder / "target_proportions.npz")["arr_0"]
uxm_train_data = np.load(uxm_pred_folder / "uxm_results_train.npz")["arr_0"]
uxm_val_data = np.load(uxm_pred_folder / "uxm_results_valid.npz")["arr_0"]
uxm_test_data = np.load(uxm_pred_folder / "uxm_results_test.npz")["arr_0"]

print(
    f"Train: {uxm_train_data.shape}, Val: {uxm_val_data.shape}, Test: {uxm_test_data.shape}"
)
print(f"Targets: {target_prop.shape}")

Train: (100000, 39), Val: (100000, 39), Test: (100000, 39)
Targets: (100000, 39)


## Evaluation helper

In [4]:
def evaluate_test(test_pred: dict, round_: int = 6) -> pd.DataFrame:
    """Compute deconvolution metrics for each method and return a sorted DataFrame."""
    results = {
        name: compute_deconvolution_metrics(
            pred=pred, target=target_prop, class_names=ctype_names_list
        )
        for name, pred in test_pred.items()
    }
    # Drop per-class arrays for compact display
    results = {
        k: {m: v for m, v in metrics.items() if "per_class" not in m}
        for k, metrics in results.items()
    }
    df = pd.DataFrame(results).T
    float_cols = [
        "mae",
        "mse",
        "kl",
        "max_error",
        "cosine_sim",
        "loa_lower",
        "loa_upper",
        "loa_width",
        "worst_class_loa_lower",
        "worst_class_loa_upper",
        "worst_class_loa_width",
    ]
    for col in float_cols:
        if col in df.columns:
            df[col] = df[col].astype(float).round(round_)
    return df.sort_values("mse")

## 1. Affine (Linear) calibrator

In [6]:
linear_cal = LinearCalibrator()
linear_cal.fit(uxm_val_data, target_prop)

# Predict with both normalization strategies
uxm_test_lin_clip_norm = linear_cal.predict(
    uxm_test_data, norm_method="clip0-normalize"
)
uxm_test_lin_simplex = linear_cal.predict(
    uxm_test_data, norm_method="simplex-projection"
)

## 2. Vector Scaling calibrator — grid search

In [ ]:
param_grid = {
    "reg_lambda": [0.0, 1e-4, 1e-3, 1e-2, 1e-1],
    "lr": [1e-2],
    "max_iter": [500],
    "scheduler": ["cosine"],
    "batch_size": [None],  # full-batch
    "patience": [50],
    "tol": [1e-7],
}


def prod_len(d: dict) -> int:
    return reduce(lambda x, y: x * y, (len(v) for v in d.values()), 1)


results_gs = []
best_val_loss = np.inf
best_vs_model: VectorScalingCalibrator | None = None

n_combos = prod_len(param_grid)
with tqdm(total=n_combos, desc="VS grid search") as pbar:
    for reg_lambda in param_grid["reg_lambda"]:
        for lr in param_grid["lr"]:
            for max_iter in param_grid["max_iter"]:
                for scheduler in param_grid["scheduler"]:
                    for batch_size in param_grid["batch_size"]:
                        for patience in param_grid["patience"]:
                            for tol in param_grid["tol"]:
                                vs_cal = VectorScalingCalibrator(
                                    reg_lambda=reg_lambda,
                                    optimizer="adam",
                                    lr=lr,
                                    scheduler=scheduler,
                                    max_iter=max_iter,
                                    batch_size=batch_size,
                                    patience=patience,
                                    tol=tol,
                                    device="cuda",
                                    verbose=False,
                                )
                                vs_cal.fit(
                                    X=uxm_train_data,
                                    y=target_prop,
                                    X_val=uxm_val_data,
                                    y_val=target_prop,
                                )
                                results_gs.append(
                                    {
                                        "reg_lambda": reg_lambda,
                                        "lr": lr,
                                        "max_iter": max_iter,
                                        "scheduler": scheduler,
                                        "batch_size": batch_size,
                                        "patience": patience,
                                        "tol": tol,
                                        "best_train_loss": vs_cal.best_metrics_[
                                            "train_loss"
                                        ],
                                        "best_val_loss": vs_cal.best_metrics_[
                                            "val_loss"
                                        ],
                                        "best_train_mse": vs_cal.best_metrics_[
                                            "train_mse"
                                        ],
                                        "best_val_mse": vs_cal.best_metrics_["val_mse"],
                                        "best_epoch": vs_cal.best_epoch_,
                                    }
                                )
                                if vs_cal.best_metrics_["val_loss"] < best_val_loss:

                                    best_val_loss = vs_cal.best_metrics_["val_loss"]
                                    best_vs_model = vs_cal
                                pbar.update(1)

gs_df = pd.DataFrame(results_gs).sort_values("best_val_loss")
gs_df

In [ ]:
# actual kfold grid search with VectorScalingCalibrator with CV
vs_cal_cv = CrossValidationEngine()
vs_cal_cv.fit(
    X=uxm_train_data,
    y=target_prop,
    X_val=uxm_val_data,
    y_val=target_prop,
    model_class=VectorScalingCalibrator,
    model_param_grid={
        "reg_lambda": [0.0, 1e-4],
        "lr": [1e-4, 1e-3, 1e-2],
        "max_iter": [1000, 1500, 2000],
        "optimizer": ["adam"],
        "scheduler": ["plateau"],
        "patience": [20],
        "tol": [1e-4],
    },
    n_folds=3,
)

CV grid search:   0%|          | 0/54 [00:00<?, ?it/s]

CV grid search: 100%|██████████| 54/54 [00:33<00:00,  1.63it/s, best_metric=1.4465438e+00, best_params={'lr': 0.01, 'max_iter': 1000, 'optimizer': 'adam', 'patience': 20, 'reg_lambda': 0.0, 'scheduler': 'plateau', 'tol': 0.0001}]  


Epoch -1: train_loss = 1.4699713e+00, train_mse = 9.4913259e-05
Epoch 0: train_loss = 1.4699713e+00, train_mse = 8.8394740e-05
Early stopping at epoch 90 (best loss: 1.4460649e+00)


In [ ]:
rows = []
for entry in [
    k + tuple(v.items()) for k, v in vs_cal_cv.best_metrics_per_param_.items()
]:
    row = {}
    lr_count = 0
    for key, value in entry:
        if key == "lr":
            row["lr_init" if lr_count == 0 else "lr_final"] = value
            lr_count += 1
        else:
            row[key] = value
    rows.append(row)

df = pd.DataFrame(rows)
df.sort_values("val_loss")

## 3. Test-set evaluation

In [ ]:
vs_cal_cv.predict()

In [ ]:
uxm_test_vs = vs_cal_cv.predict(uxm_test_data)

results_df = evaluate_test(
    {
        "UXM (no calibration)": uxm_test_data,
        "Affine + clip0-normalize": uxm_test_lin_clip_norm,
        "Affine + simplex-projection": uxm_test_lin_simplex,
        "Vector Scaling (best)": uxm_test_vs,
    }
)
results_df.sort_values("mse")